In [ ]:
import os
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from playwright.sync_api import sync_playwright

# Carica le variabili da .env
load_dotenv()
USERNAME = os.getenv("LOGIN_HOLISTIC")
PASSWORD = os.getenv("PASSWORD_HOLISTIC")

BASE_URL = ""https://holistic.csirt.local"
LOGIN_URL = f"{BASE_URL}/Login"  # Inserisci l'URL di login
#TARGET_URL = "https://esempio.com/dati"  # Inserisci l'URL con i dati da estrarre
TARGET_GUID = "0e4c4d47-e368-48a3-9b53-9521c666a2cc"
TARGET_URL = f"{BASE_URL}/Private/Soggetto?GUID={TARGET_GUID}"

with sync_playwright() as p:
    # headless=False apre la finestra del browser per vedere cosa succede
    browser = p.chromium.launch(headless=False, slow_mo=300)
    context = browser.new_context()
    page = context.new_page()

    print("1. Caricamento pagina di login...")
    page.goto(LOGIN_URL)

    # 2. Compilazione Username tramite etichetta o placeholder
    print("2. Inserimento credenziali...")
    try:
        page.get_by_label("Nome Utente", exact=False).fill(USERNAME)
    except Exception:
        page.get_by_placeholder("Nome Utente", exact=False).fill(USERNAME)

    # 3. Compilazione Password
    try:
        page.get_by_label("Password", exact=False).fill(PASSWORD)
    except Exception:
        page.get_by_placeholder("Password", exact=False).fill(PASSWORD)

    # 4. Clic sul pulsante di invio
    print("3. Invio form...")
    try:
        # Cerca pulsanti tipici ("Accedi", "Login", "Entra", "Invia")
        page.locator(
            "button:has-text('Accedi'), button:has-text('Login'), button:has-text('Entra'), input[type='submit']"
        ).first.click()
    except Exception:
        # Fallback: preme Invio direttamente nel campo password
        page.keyboard.press("Enter")

    # 5. Attesa del completamento del caricamento post-login
    page.wait_for_load_state("networkidle")
    print("4. Login effettuato! URL corrente:", page.url)

    # 6. Spostamento alla pagina con i dati (se diversa dalla landing)
    if TARGET_URL and page.url != TARGET_URL:
        page.goto(TARGET_URL)
        page.wait_for_load_state("networkidle")

    # 7. Recupero dell'HTML e parsing iniziale con BeautifulSoup
    html_markup = page.content()
    soup = BeautifulSoup(html_markup, "html.parser")
    print(
        f"5. Dati scaricati con successo ({len(html_markup)} caratteri HTML)."
    )

    # Lascia aperta la finestra finché non premi Invio nel terminale
    input(
        "\nPremi INVIO nel terminale di VSCode per chiudere il browser..."
    )
    browser.close()